In [6]:
import os
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------
# Helpers
# ---------------------------------------------------

def sanity_check(adata, color=None, name="adata"):
    print("\n======================")
    print(f"🔍   ANNDATA CHECK ({name})")
    print("======================")
    print(f"Shape: {adata.n_obs:,} cells × {adata.n_vars:,} genes")
    print("obs columns:", list(adata.obs.columns))
    print("obsm keys:", list(adata.obsm.keys()))

    if color is not None:
        if color in adata.obs:
            print(f"\nColumn '{color}': {adata.obs[color].nunique()} unique values")
            print(f"Missing: {adata.obs[color].isna().sum():,}")
        else:
            print(f"\n⚠️ Column '{color}' NOT FOUND in obs")

    if "X_umap" in adata.obsm:
        print("UMAP present:", adata.obsm["X_umap"].shape)
    else:
        print("No UMAP yet")
    print("======================\n")


def neighbors_umap_on_embedding(adata, use_rep="concept_mean_embedding", n_pcs=50, min_dist=0.3):
    print(f"⚙️ Using embedding key: {use_rep}")
    if use_rep not in adata.obsm:
        raise KeyError(f"{use_rep} not found in adata.obsm")

    emb = adata.obsm[use_rep]

    print("📉 Running PCA on embeddings…")
    X_pca = sc.tl.pca(emb, n_comps=n_pcs, svd_solver="arpack", copy=True)
    adata.obsm["X_pca_emb"] = X_pca

    print("👥 Computing neighbors…")
    sc.pp.neighbors(adata, use_rep="X_pca_emb")

    print("🌀 Running UMAP…")
    sc.tl.umap(adata, min_dist=min_dist, random_state=0)


def plot_umap(adata, color, palette=None, title=None, out_png=None):
    if title is None:
        title = f"UMAP colored by {color}"

    print(f"🎨 Plotting UMAP colored by '{color}'…")
    fig = sc.pl.umap(
        adata,
        color=color,
        frameon=False,
        palette=palette,
        title=title,
        show=False,
        return_fig=True,
    )
    plt.show()

    if out_png is not None:
        fig.savefig(out_png, dpi=300, bbox_inches="tight")
        print(f"💾 Saved: {out_png}")


# ---------------------------------------------------
# Color palette for cell types
# ---------------------------------------------------
cell_type_palette = {
    'ABCs': '#023fa5',
    'Astrocytes': '#7d87b9',
    'Astroependymal': '#bec1d4',
    'BAMs': '#d6bcc0',
    'Bergmann': '#bb7784',
    'Choroid-Plexus': '#8e063b',
    'ECs': '#4a6fe3',
    'Ependymal': '#8595e1',
    'Immune-Other': '#b5bbe3',
    'Microglia': '#e6afb9',
    'Neurons-Dopa': '#e07b91',
    'Neurons-Gaba': '#d33f6a',
    'Neurons-Glut': '#11c638',
    'Neurons-Glyc-Gaba': '#8dd593',
    'Neurons-Granule-Immature': '#c6dec7',
    'Neurons-Other': '#ead3c6',
    'OECs': '#f0b98d',
    'OPCs': '#ef9708',
    'Oligodendrocytes': '#0fcfc0',
    'Pericytes': '#9cded6',
    'SMCs': '#d5eae7',
    'Tanycytes': '#f3e1eb',
    'Undefined': '#f6c4e1',
    'VLMCs': '#f79cd4',
    "Unknown": "#aaaaaa",
}


In [7]:
# ---------------------------------------------------
# LOAD EMBEDDINGS + ANNOTATIONS
# ---------------------------------------------------

ZHUANG_PATH = "/p/project1/hai_fzj_bda/spitzer2/point_transformer/data/raw/Zhuang-ABCA-1_concept_embeddings.h5ad"
ZENG_PATH   = "/p/project1/hai_fzj_bda/spitzer2/point_transformer/data/raw/Zeng_concept_embeddings.h5ad"
ISD_PATH    = "/p/project1/hai_fzj_bda/spitzer2/point_transformer/data/processed/ISD-1_concept_embeddings.h5ad"

ZHUANG_ANN_PATH = "/p/project1/hai_fzj_bda/salg1/cellseg-benchmark/data_dir/samples/zhuang/results/merfish/cell_type_annotation/adata_obs_annotated.csv"
ZENG_ANN_PATH   = "/p/project1/hai_fzj_bda/salg1/cellseg-benchmark/data_dir/samples/zeng/results/merfish/cell_type_annotation/adata_obs_annotated.csv"

SUBSAMPLE = 100_000

In [13]:
# ---------------------------------------------------
# Load ZHUANG
# ---------------------------------------------------
print(f"📂 Loading Zhuang: {ZHUANG_PATH}")
zhuang = sc.read(ZHUANG_PATH)
zhuang.obs_names = zhuang.obs_names.astype(str)
zhuang.obs["cell_id_raw"] = zhuang.obs_names
zhuang.obs_names = zhuang.obs["cell_id_raw"] + "-Zhuang"
zhuang.obs["cell_id"] = zhuang.obs_names
sc.pp.subsample(zhuang, n_obs=SUBSAMPLE, random_state=0)

sanity_check(zhuang, name="Zhuang")
# ---------------------------------------------------
# Load ZENG
# ---------------------------------------------------
print(f"📂 Loading Zeng: {ZENG_PATH}")
zeng = sc.read(ZENG_PATH)
zeng.obs_names = zeng.obs_names.astype(str)
zeng.obs["cell_id_raw"] = zeng.obs_names
zeng.obs_names = zeng.obs["cell_id_raw"] + "-Zeng"
zeng.obs["cell_id"] = zeng.obs_names
sc.pp.subsample(zeng, n_obs=SUBSAMPLE, random_state=0)

sanity_check(zeng, name="Zeng")

📂 Loading Zhuang: /p/project1/hai_fzj_bda/spitzer2/point_transformer/data/raw/Zhuang-ABCA-1_concept_embeddings.h5ad

🔍   ANNDATA CHECK (Zhuang)
Shape: 100,000 cells × 1,122 genes
obs columns: ['abc_sample_id', 'brain_section_label', 'brain_section_label_adata', 'class', 'class_color', 'cluster', 'cluster_alias', 'cluster_color', 'cluster_confidence_score', 'donor_genotype', 'donor_label', 'donor_sex', 'feature_matrix_label', 'high_quality_transfer', 'neurotransmitter', 'neurotransmitter_color', 'parcellation_category', 'parcellation_category_color', 'parcellation_division', 'parcellation_division_color', 'parcellation_index', 'parcellation_organ', 'parcellation_organ_color', 'parcellation_structure', 'parcellation_structure_color', 'parcellation_substructure', 'parcellation_substructure_color', 'subclass', 'subclass_color', 'subclass_confidence_score', 'supertype', 'supertype_color', 'x', 'x_ccf', 'y', 'y_ccf', 'z', 'z_ccf', 'cell_id_raw', 'cell_id']
obsm keys: ['concept_cls_embedding'

In [20]:
print("\n🔍 Checking Zeng ID structure…")

# 1. Extract base ID inside .obs (correct way)
zeng.obs["base_id"] = zeng.obs["cell_id"].str.replace(
    r"-\d+(?=-Zeng$)", "", regex=True
)

# 2. Count how many had suffixes (-1, -2, …)
num_with_suffix = (zeng.obs["cell_id"] != zeng.obs["base_id"] + "-Zeng").sum()
print(f"Cells with '-1'/'-2' suffixes: {num_with_suffix:,}")

# 3. Check for duplicated base IDs (multiple fragments)
dups = zeng.obs["base_id"][zeng.obs["base_id"].duplicated(keep=False)]
print(f"Base IDs appearing more than once: {dups.nunique():,}")

# 4. Show examples
if len(dups) > 0:
    print("\n📌 Example multi-fragment IDs:")
    for base in dups.unique()[:10]:
        variants = zeng.obs["cell_id"][zeng.obs["base_id"] == base].tolist()
        print(f"{base}: {variants}")
else:
    print("\n✅ No fragment suffixes detected — SAFE, nothing to strip")



🔍 Checking Zeng ID structure…
Cells with '-1'/'-2' suffixes: 100,000
Base IDs appearing more than once: 1,423

📌 Example multi-fragment IDs:
1018093344101150510-Zeng: ['1018093344101150510-Zeng', '1018093344101150510-1-Zeng']
1104095349100570323-Zeng: ['1104095349100570323-Zeng', '1104095349100570323-1-Zeng']
1018093344102620322-Zeng: ['1018093344102620322-3-Zeng', '1018093344102620322-1-Zeng', '1018093344102620322-Zeng']
1018093344101980294-Zeng: ['1018093344101980294-4-Zeng', '1018093344101980294-1-Zeng']
1018093344101340319-Zeng: ['1018093344101340319-1-Zeng', '1018093344101340319-3-Zeng']
1018093344100780684-Zeng: ['1018093344100780684-1-Zeng', '1018093344100780684-Zeng']
1018093344101840266-Zeng: ['1018093344101840266-5-Zeng', '1018093344101840266-2-Zeng']
1018093344102160232-Zeng: ['1018093344102160232-4-Zeng', '1018093344102160232-2-Zeng']
1018093344102210368-Zeng: ['1018093344102210368-2-Zeng', '1018093344102210368-3-Zeng']
1018093344100970462-Zeng: ['1018093344100970462-5-Zen

In [16]:
def print_cell_ids(adata, name, n=10):
    print(f"\n📌 {name} — FIRST {n} OBS NAMES:")
    print(adata.obs_names[:n].tolist())

    if "cell_id" in adata.obs:
        print(f"📌 {name} — FIRST {n} cell_id:")
        print(adata.obs['cell_id'][:n].tolist())
    else:
        print("⚠️ No 'cell_id' column in this AnnData.")

    if "cell_type" in adata.obs:
        print(f"📌 {name} — FIRST {n} cell_type:")
        print(adata.obs['cell_type'][:n].tolist())
    else:
        print("⚠️ No 'cell_type' column in this AnnData.")

    print("\n🔍 Counts:")
    if "cell_type" in adata.obs:
        print(adata.obs['cell_type'].value_counts(dropna=False).head())
    print("="*60)
print_cell_ids(zhuang, "ZHUANG")
print_cell_ids(zeng, "ZENG")



📌 ZHUANG — FIRST 10 OBS NAMES:
['196269251661640657821558596194131516996-Zhuang', '74694989265068621136814017210917662188-Zhuang', '153481091143684790537764283973536365605-Zhuang', '35754669376327154389380727082032974607-Zhuang', '209284113510049681080388547425982874009-Zhuang', '297432916244828122182771714624565340927-Zhuang', '246821969892156900584746899748023911665-Zhuang', '195019211204716016282645476521166239247-Zhuang', '186322290372526148915740912556582003868-Zhuang', '224696648076438278786182882631045664437-Zhuang']
📌 ZHUANG — FIRST 10 cell_id:
['196269251661640657821558596194131516996-Zhuang', '74694989265068621136814017210917662188-Zhuang', '153481091143684790537764283973536365605-Zhuang', '35754669376327154389380727082032974607-Zhuang', '209284113510049681080388547425982874009-Zhuang', '297432916244828122182771714624565340927-Zhuang', '246821969892156900584746899748023911665-Zhuang', '195019211204716016282645476521166239247-Zhuang', '186322290372526148915740912556582003868-

In [21]:
# Load annotation
z_ann = pd.read_csv(ZHUANG_ANN_PATH)
z_ann["base_id"] = z_ann["cell_id"].astype(str).str.split("-").str[0]
z_ann["cell_id_combined"] = z_ann["base_id"] + "-Zhuang"
z_ann = z_ann.rename(columns={"cell_type_mmc_raw": "cell_type"})

print("\n📄 ZHUANG ANNOTATION (cell_id_combined, cell_type):")
print(z_ann[["cell_id_combined", "cell_type"]].head(20))
print(f"Total rows: {len(z_ann):,}")


# Load annotation
ze_ann = pd.read_csv(ZENG_ANN_PATH)
ze_ann["cell_id_combined"] = ze_ann["cell_id"].astype(str) + "-Zeng"
ze_ann = ze_ann.rename(columns={"cell_type_mmc_raw": "cell_type"})

print("\n📄 ZENG ANNOTATION (cell_id_combined, cell_type):")
print(ze_ann[["cell_id_combined", "cell_type"]].head(20))
print(f"Total rows: {len(ze_ann):,}")


📄 ZHUANG ANNOTATION (cell_id_combined, cell_type):
                                  cell_id_combined     cell_type
0    38542084966093731202691164843098659276-Zhuang  Neurons-Glut
1   183256764301191388575899679882178798208-Zhuang  Neurons-Glut
2    37353846072841168055577846846204745147-Zhuang  Neurons-Glut
3    56522635880108955114956785931951865125-Zhuang  Neurons-Glut
4   114075972918310011122352881571035240931-Zhuang  Neurons-Glut
5   161395906901535153611343134398807012103-Zhuang  Neurons-Glut
6   196430079481818150765904306976008332942-Zhuang  Neurons-Glut
7   301630835670385691349511479183791495268-Zhuang  Neurons-Glut
8   118531435515176328294928752438827327938-Zhuang  Neurons-Glut
9   187669921940761829522139776202588300515-Zhuang  Neurons-Glut
10  320963102701035973479619920837315703378-Zhuang  Neurons-Glut
11   69288500551118823610104206177484494301-Zhuang  Neurons-Glut
12   95054221102774891854992075933030591651-Zhuang  Neurons-Glut
13   321618764101676563422956261780591

/tmp/ipykernel_43899/3244544377.py:13: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  ze_ann = pd.read_csv(ZENG_ANN_PATH)



📄 ZENG ANNOTATION (cell_id_combined, cell_type):
            cell_id_combined      cell_type
0   1019171911101460569-Zeng   Neurons-Glut
1   1019171911101550321-Zeng   Neurons-Glut
2   1019171911100841066-Zeng   Neurons-Glut
3   1019171911101400425-Zeng  Neurons-Other
4   1019171911101380264-Zeng   Neurons-Glut
5   1019171911101130445-Zeng  Neurons-Other
6   1019171911101280013-Zeng  Neurons-Other
7   1019171911101120478-Zeng  Neurons-Other
8   1019171911101130255-Zeng  Neurons-Other
9   1019171911101110358-Zeng  Neurons-Other
10  1019171911101540031-Zeng   Neurons-Glut
11  1019171911101450426-Zeng   Neurons-Glut
12  1019171911101130343-Zeng  Neurons-Other
13  1019171911101120455-Zeng  Neurons-Other
14  1019171911101120292-Zeng  Neurons-Other
15  1019171911101130410-Zeng  Neurons-Other
16  1019171911101130177-Zeng  Neurons-Other
17  1019171911101120409-Zeng  Neurons-Other
18  1019171911101110571-Zeng  Neurons-Other
19  1019171911101270045-Zeng  Neurons-Other
Total rows: 3,739,961


In [22]:
zhuang.obs = zhuang.obs.merge(
    z_ann[["cell_id_combined", "cell_type"]],
    left_on="cell_id",
    right_on="cell_id_combined",
    how="left"
).drop(columns=["cell_id_combined"])

zhuang.obs["dataset"] = "Zhuang"
sanity_check(zhuang, color="cell_type", name="Zhuang")


zeng.obs = zeng.obs.merge(
    ze_ann[["cell_id_combined", "cell_type"]],
    left_on="cell_id",
    right_on="cell_id_combined",
    how="left"
).drop(columns=["cell_id_combined"])

zeng.obs["dataset"] = "Zeng"
sanity_check(zeng, color="cell_type", name="Zeng")


🔍   ANNDATA CHECK (Zhuang)
Shape: 100,000 cells × 1,122 genes
obs columns: ['abc_sample_id', 'brain_section_label', 'brain_section_label_adata', 'class', 'class_color', 'cluster', 'cluster_alias', 'cluster_color', 'cluster_confidence_score', 'donor_genotype', 'donor_label', 'donor_sex', 'feature_matrix_label', 'high_quality_transfer', 'neurotransmitter', 'neurotransmitter_color', 'parcellation_category', 'parcellation_category_color', 'parcellation_division', 'parcellation_division_color', 'parcellation_index', 'parcellation_organ', 'parcellation_organ_color', 'parcellation_structure', 'parcellation_structure_color', 'parcellation_substructure', 'parcellation_substructure_color', 'subclass', 'subclass_color', 'subclass_confidence_score', 'supertype', 'supertype_color', 'x', 'x_ccf', 'y', 'y_ccf', 'z', 'z_ccf', 'cell_id_raw', 'cell_id', 'cell_type', 'dataset']
obsm keys: ['concept_cls_embedding', 'concept_mean_embedding']

Column 'cell_type': 23 unique values
Missing: 37,394
No UMAP ye

In [23]:
# ---------------------------------------------------
# LOAD ISD (NO ANNOTATION)
# ---------------------------------------------------
print(f"📂 Loading ISD: {ISD_PATH}")
isd = sc.read(ISD_PATH)

# Ensure string index
isd.obs_names = isd.obs_names.astype(str)

# Proper cell_id: use the row index (same as Zhuang/Zeng)
isd.obs["cell_id"] = isd.obs_names

# Proper cell type: use the annotation column that Josef said is present
if "cell_type_mmc_raw_revised" in isd.obs:
    isd.obs["cell_type"] = isd.obs["cell_type_mmc_raw_revised"]
else:
    raise KeyError("ISD is missing 'cell_type_mmc_raw_revised' annotation!")

# Add dataset identifier
isd.obs["dataset"] = "ISD"

sanity_check(isd, color="cell_type", name="ISD")


📂 Loading ISD: /p/project1/hai_fzj_bda/spitzer2/point_transformer/data/processed/ISD-1_concept_embeddings.h5ad

🔍   ANNDATA CHECK (ISD)
Shape: 46,169 cells × 500 genes
obs columns: ['region', 'slide', 'cell_id', 'area', 'n_counts', 'n_genes', 'cell_type_mmc_raw_revised', 'acronym', 'low_quality_cell', 'area_outlier_cell', 'cell_type', 'dataset']
obsm keys: ['X_pca', 'X_umap', 'concept_cls_embedding', 'concept_mean_embedding', 'intensities', 'spatial']

Column 'cell_type': 19 unique values
Missing: 0
UMAP present: (46169, 2)



In [25]:
import anndata as ad
def strip_obsm_indices(adata):
    for k in list(adata.obsm.keys()):
        arr = adata.obsm[k]
        # If it’s a DataFrame, convert to numpy
        try:
            adata.obsm[k] = arr.to_numpy()
        except AttributeError:
            # Already numpy, nothing to do
            pass

# Apply to all datasets
for A in [zhuang, zeng, isd]:
    strip_obsm_indices(A)

# ---------------------------------------------------
# CONCATENATE ALL 3 DATASETS
# ---------------------------------------------------
print("🔗 Concatenating Zhuang + Zeng + ISD …")
combined = ad.concat(
    [zhuang, zeng, isd],
    label="source",
    join="outer",   # <— THE CRITICAL FIX
    index_unique=None
)


combined.obs_names_make_unique()
sanity_check(combined, color="dataset", name="Combined")


# ---------------------------------------------------
# RUN UMAP ON EMBEDDINGS
# ---------------------------------------------------
neighbors_umap_on_embedding(
    combined,
    use_rep="concept_mean_embedding",
    n_pcs=50,
    min_dist=0.3,
)


🔗 Concatenating Zhuang + Zeng + ISD …


/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")



🔍   ANNDATA CHECK (Combined)
Shape: 246,169 cells × 1,695 genes
obs columns: ['abc_sample_id', 'brain_section_label', 'brain_section_label_adata', 'class', 'class_color', 'cluster', 'cluster_alias', 'cluster_color', 'cluster_confidence_score', 'donor_genotype', 'donor_label', 'donor_sex', 'feature_matrix_label', 'high_quality_transfer', 'neurotransmitter', 'neurotransmitter_color', 'parcellation_category', 'parcellation_category_color', 'parcellation_division', 'parcellation_division_color', 'parcellation_index', 'parcellation_organ', 'parcellation_organ_color', 'parcellation_structure', 'parcellation_structure_color', 'parcellation_substructure', 'parcellation_substructure_color', 'subclass', 'subclass_color', 'subclass_confidence_score', 'supertype', 'supertype_color', 'x', 'x_ccf', 'y', 'y_ccf', 'z', 'z_ccf', 'cell_id_raw', 'cell_id', 'cell_type', 'dataset', 'average_correlation_score', 'base_id', 'region', 'slide', 'area', 'n_counts', 'n_genes', 'cell_type_mmc_raw_revised', 'acron

In [26]:
# ---------------------------------------------------
# PLOTS
# ---------------------------------------------------
cwd = os.getcwd()

# 1) UMAP colored by dataset
plot_umap(
    combined,
    color="dataset",
    title="UMAP (dataset)",
)

# 2) UMAP colored by cell_type
plot_umap(
    combined,
    color="cell_type",
    palette=cell_type_palette,
    title="UMAP (cell_type)",
)

print("\n🎉 DONE!")


🎨 Plotting UMAP colored by 'dataset'…
🎨 Plotting UMAP colored by 'cell_type'…

🎉 DONE!


In [27]:
import os
print(combined.obs_names.is_unique)

# Must set BEFORE importing scib
os.environ["TMPDIR"] = "/p/scratch/cjinm16/dipippo1/tmp_lisi"
os.makedirs(os.environ["TMPDIR"], exist_ok=True)

import scib
import scanpy as sc



print("Computing neighbors on subsampled data…")
sc.pp.neighbors(
    combined,
    use_rep="concept_cls_embedding",   # MUST point to a valid obsm key
    n_neighbors=30
)

print("Running cLISI…")
clisi = scib.metrics.clisi_graph(
    combined,
    label_key="dataset",
    type_="knn",
)

print("cLISI:", clisi)

True
Computing neighbors on subsampled data…
Running cLISI…
cLISI: 0.684415551641665


In [28]:
from scib_metrics.benchmark import Benchmarker

# 1) Ensure annotation column is string
combined.obs["cell_type_string"] = combined.obs["cell_type"].astype(str)

# 2) Run Benchmarker on your embedding
bm = Benchmarker(
    combined,
    batch_key="dataset",                 # THIS IS THE CORRECT BATCH COLUMN
    label_key="cell_type_string",        # your cell type
    embedding_obsm_keys=["concept_cls_embedding"],  # or "concept_mean_embedding"
    n_jobs=-1,
)

# 3) Benchmark
bm.benchmark()

# 4) Print results
results = bm.get_results()
print(results)


/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/scanpy/preprocessing/_pca/__init__.py:226: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(
rics:   0%|          | 0/10 [00:00<?, ?it/s]

rics:  60%|██████    | 6/10 [1:02:51<21:28, 322.00s/it, Batch correction: kbet_per_label]

INFO     Undefined consists of a single batch or is too small. Skip.                                               



rics:  70%|███████   | 7/10 [1:08:35<16:28, 329.43s/it, Batch correction: kbet_per_label]
/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/scib_metrics/metrics/_graph_connectivity.py:32: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  tab = pd.value_counts(comps)

Embeddings: 100%|██████████| 1/1 [1:08:39<00:00, 4119.70s/it]tch correction: pcr_comparison]

                                                                                         

                        Isolated labels        KMeans NMI        KMeans ARI  \
Embedding                                                                     
concept_cls_embedding          0.398005          0.309087          0.105639   
Metric Type            Bio conservation  Bio conservation  Bio conservation   

                       Silhouette label             cLISI              BRAS  \
Embedding                                                                     
concept_cls_embedding          0.464263          0.973181           0.71207   
Metric Type            Bio conservation  Bio conservation  Batch correction   

                                  iLISI              KBET Graph connectivity  \
Embedding                                                                      
concept_cls_embedding          0.204337          0.225105           0.619118   
Metric Type            Batch correction  Batch correction   Batch correction   

                         PCR comparison Batch

In [29]:
print("Running cLISI…")
clisi = scib.metrics.clisi_graph(
    combined,
    label_key="cell_type_string",
    type_="knn",
)
print("cLISI:", clisi)

Running cLISI…
cLISI: 0.9666968243534827


In [30]:
print("cLISI:", clisi)

cLISI: 0.9666968243534827
